# Assignment 1: Linear Regression (From Scratch) - Teens Mental Health Dataset

This notebook implements **Linear Regression from scratch** using only NumPy, without using scikit-learn's built-in regressor. We use Gradient Descent optimization to find the optimal weights that minimize the Mean Squared Error (MSE).

## Dataset
- **Teens Mental Health Dataset**
- Target: academic_performance (GPA-like score)
- Features: age, gender, daily_social_media_hours, platform_usage, sleep_hours, screen_time_before_sleep, physical_activity, social_interaction_level, stress_level, anxiety_level, addiction_level

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 1. Load and Explore the Dataset

In [ ]:
# Load Teens Mental Health Dataset
df = pd.read_csv("dataset/Teen_Mental_Health_Dataset.csv")
print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
df.head()

## 2. Data Preprocessing

### 2.1 Check for Missing Values

In [ ]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nTotal missing values: {df.isnull().sum().sum()}")

### 2.2 Handle Categorical Variables

In [ ]:
# Handle categorical variables - encode them numerically
df_encoded = df.copy()

# Label encode categorical columns
label_encoders = {}
categorical_columns = ['gender', 'platform_usage', 'social_interaction_level']

for col in categorical_columns:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])
    label_encoders[col] = le

print("Categorical encoding completed!")
for col, le in label_encoders.items():
    print(f"{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

### 2.3 Feature Selection and Train-Test Split

In [ ]:
# Feature Selection - predict academic_performance
X = df_encoded.drop('academic_performance', axis=1)
y = df_encoded['academic_performance']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"\nTraining samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")

### 2.4 Feature Scaling (Standard Normalization)

In [ ]:
# Normalization using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("After scaling:")
print(f"X_train mean (should be ~0): {X_train_scaled.mean(axis=0)[:3]}")
print(f"X_train std (should be ~1): {X_train_scaled.std(axis=0)[:3]}")

## 3. Linear Regression Implementation (From Scratch)

### 3.1 Theory

Linear Regression using Gradient Descent:
- Hypothesis: h(x) = X * theta (matrix multiplication)
- Cost Function (MSE): J(theta) = (1/2m) * sum((h(x) - y)^2)
- Gradient: grad J(theta) = (1/m) * X^T * (X*theta - y)
- Update: theta = theta - alpha * grad J(theta)

where alpha is the learning rate.

In [ ]:
class LinearRegressionFromScratch:
    """
    Linear Regression implemented from scratch using Gradient Descent.
    This implementation does NOT use sklearn SGDRegressor - it uses pure NumPy.
    """
    def __init__(self, learning_rate=0.01, n_epochs=1000, regularization=0.0):
        self.learning_rate = learning_rate
        self.n_epochs = n_epochs
        self.regularization = regularization
        self.theta = None
        self.loss_history = []
    
    def _add_bias(self, X):
        return np.c_[np.ones(X.shape[0]), X]
    
    def fit(self, X, y):
        m, n = X.shape
        X_b = self._add_bias(X)
        self.theta = np.zeros(n + 1)
        
        for epoch in range(self.n_epochs):
            predictions = X_b @ self.theta
            error = predictions - y
            gradient = (1/m) * (X_b.T @ error)
            gradient[1:] += (self.regularization / m) * self.theta[1:]
            self.theta -= self.learning_rate * gradient
            mse = (1/(2*m)) * np.sum(error**2)
            self.loss_history.append(mse)
        return self
    
    def predict(self, X):
        X_b = self._add_bias(X)
        return X_b @ self.theta

print("LinearRegressionFromScratch class defined!")

### 3.2 Train the Model

In [ ]:
# Create and train the model
model = LinearRegressionFromScratch(
    learning_rate=0.01, 
    n_epochs=1000, 
    regularization=0.0
)

# Train on scaled training data
model.fit(X_train_scaled, y_train)

print(f"Training completed!")
print(f"Final training MSE: {model.loss_history[-1]:.4f}")
print(f"Number of epochs: {len(model.loss_history)}")

### 3.3 Visualize Loss Curve

In [ ]:
# Plot Loss Curve
plt.figure(figsize=(10, 6))
plt.plot(range(model.n_epochs), model.loss_history, 'b-', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Mean Squared Error (MSE)', fontsize=12)
plt.title('Training Loss Curve - Linear Regression (From Scratch)', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('screenshots/Loss_Curve.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nObservations:")
print(f"- Initial MSE: {model.loss_history[0]:.4f}")
print(f"- Final MSE: {model.loss_history[-1]:.4f}")
print(f"- MSE Reduction: {((model.loss_history[0] - model.loss_history[-1])/model.loss_history[0]*100):.2f}%")

## 4. Model Evaluation

In [ ]:
# Predict on test set
y_pred = model.predict(X_test_scaled)

# Calculate evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("=" * 50)
print("LINEAR REGRESSION MODEL EVALUATION (From Scratch)")
print("=" * 50)
print(f"Mean Absolute Error (MAE):       {mae:.4f}")
print(f"Mean Squared Error (MSE):        {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"R-squared (R2):                   {r2:.4f}")
print("=" * 50)

if r2 > 0.7:
    print("\nGood model fit! The model explains more than 70% of variance.")
elif r2 > 0.5:
    print("\nModerate model fit. Consider tuning hyperparameters.")
else:
    print("\nPoor model fit. Try adjusting learning rate or epochs.")

### 4.1 Actual vs Predicted Comparison

In [ ]:
# Sample predictions comparison
sample_indices = range(10)
sample_df = pd.DataFrame({
    'Actual Value': y_test.iloc[sample_indices].values,
    'Predicted Value': y_pred[sample_indices],
    'Error': y_test.iloc[sample_indices].values - y_pred[sample_indices]
})
print("Sample Predictions vs Actual Values:")
display(sample_df)

# Scatter plot of Actual vs Predicted
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.3, edgecolors='k', linewidth=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Values', fontsize=12)
plt.ylabel('Predicted Values', fontsize=12)
plt.title('Actual vs Predicted Values', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('screenshots/Actual_Predicted_Graph.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.2 Residual Analysis

In [ ]:
# Residual analysis
residuals = y_test.values - y_pred

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.hist(residuals, bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Residual (Actual - Predicted)', fontsize=11)
plt.ylabel('Frequency', fontsize=11)
plt.title('Distribution of Residuals', fontsize=12)
plt.axvline(x=0, color='r', linestyle='--', label='Zero')
plt.legend()

plt.subplot(1, 2, 2)
plt.scatter(y_pred, residuals, alpha=0.3, edgecolors='k', linewidth=0.5)
plt.axhline(y=0, color='r', linestyle='--', lw=2)
plt.xlabel('Predicted Values', fontsize=11)
plt.ylabel('Residuals', fontsize=11)
plt.title('Residual Plot', fontsize=12)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('screenshots/Residual_Analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Mean Residual: {np.mean(residuals):.4f} (should be close to 0)")
print(f"Std of Residuals: {np.std(residuals):.4f}")

## 5. Summary

This notebook demonstrated:
1. **Data Preprocessing**: Handling missing values, encoding categorical variables, train-test splitting, and feature normalization
2. **From-Scratch Implementation**: Linear Regression using pure NumPy with Gradient Descent
3. **Model Evaluation**: MAE, MSE, RMSE, R2 metrics
4. **Visualization**: Loss curve, actual vs predicted, residual analysis

The model successfully learns to predict academic performance based on teens mental health factors.